# 🚀 Production-Ready Experimental Pipeline
## 10-Class CIFAR-100 Progressive Pruning Study
### Robust, reproducible knowledge distillation with publication-quality results

## Section 0: Setup and Environment

In [ ]:
import os
import sys
import shutil
from pathlib import Path

os.chdir('/content')

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Remove old repo
if os.path.exists('/content/progressive-copying-pruning'):
    shutil.rmtree('/content/progressive-copying-pruning', ignore_errors=True)

# Clone fresh - REPLACE WITH YOUR GITHUB URL
REPO_URL = "https://github.com/YOUR_USERNAME/progressive-copying-pruning.git"
!git clone {REPO_URL}
os.chdir('/content/progressive-copying-pruning')
sys.path.insert(0, '/content/progressive-copying-pruning')

# Install dependencies
!pip install -q pyyaml omegaconf tensorboard tqdm pandas matplotlib seaborn

import torch
print("✓ Setup complete!")
print(f"✓ Working directory: {os.getcwd()}")
print(f"✓ GPU Available: {torch.cuda.is_available()}")

## Section 1: Validation Test (2 Epochs)

In [ ]:
to_clear = [key for key in sys.modules.keys() if key.startswith('src')]
for key in to_clear:
    del sys.modules[key]

from src.tasks import get_dataloaders
from src.utils.model_factory import build_classifier
import torch.nn as nn
from tqdm.notebook import tqdm
import pandas as pd
import numpy as np

print("✓ All imports successful")

In [ ]:
cfg = {
    'data': {
        'task': 'cifar100_multiclass',
        'root': './data',
        'batch_size': 128,
        'num_workers': 0,
        'selected_superclasses': [
            'aquatic_mammals', 'fish', 'flowers', 'food_containers',
            'fruit_and_vegetables', 'household_electrical',
            'household_furniture', 'insects', 'large_carnivores', 'vehicles_1'
        ]
    },
    'model': {'arch': 'resnet18_cifar', 'in_channels': 3},
    'seed': 0
}

print("[1/4] Loading data...")
train_loader, test_loader, num_classes = get_dataloaders(cfg)
print(f"✓ Train: {len(train_loader.dataset):,} samples")
print(f"✓ Test: {len(test_loader.dataset):,} samples")

In [ ]:
print("[2/4] Building model...")
model = build_classifier(cfg, num_classes)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"✓ Model: {type(model).__name__}")
print(f"✓ Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"✓ Device: {device}")

In [ ]:
print("[3/4] Testing forward pass...")
x, y = next(iter(train_loader))
x, y = x.to(device), y.to(device)
with torch.no_grad():
    output = model(x)
print(f"✓ Input: {x.shape} → Output: {output.shape}")
print(f"✓ Forward pass successful!")

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for x, y in tqdm(loader, leave=False):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (output.argmax(1) == y).sum().item()
        total += y.size(0)
    return total_loss / len(loader), 100. * correct / total

def test_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for x, y in tqdm(loader, leave=False):
            x, y = x.to(device), y.to(device)
            output = model(x)
            loss = criterion(output, y)
            total_loss += loss.item()
            correct += (output.argmax(1) == y).sum().item()
            total += y.size(0)
    return total_loss / len(loader), 100. * correct / total

print("✓ Training functions defined")

In [ ]:
print("[4/4] Quick training (2 epochs)...")
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)

for epoch in range(2):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = test_epoch(model, test_loader, criterion, device)
    print(f"  Epoch {epoch+1}: Train={train_acc:.2f}%, Test={test_acc:.2f}%")

if test_acc > 15:
    print(f"\n✅ VALIDATION PASSED! (Test: {test_acc:.2f}%)")
    print("🚀 Proceeding to full experiments...")
else:
    raise Exception("Validation failed")

## Section 2: Teacher Training (60 Epochs)

In [ ]:
print("\n" + "="*70)
print("TEACHER TRAINING - 60 Epochs (~2-3 hours)")
print("="*70)
!python -m src.main teacher --config configs/teacher/teacher_cifar100_multiclass.yaml --seed 0

In [ ]:
teacher_dir = Path('runs/teacher_cifar100_multiclass')
teacher_ckpt = teacher_dir / 'teacher.pt'
teacher_metrics = teacher_dir / 'metrics.csv'

if teacher_ckpt.exists() and teacher_metrics.exists():
    df_teacher = pd.read_csv(teacher_metrics)
    teacher_acc = df_teacher['val_acc'].max()
    print(f"\n✅ Teacher trained successfully!")
    print(f"✓ Best validation accuracy: {teacher_acc:.2f}%")
else:
    raise Exception("Teacher checkpoint not found")

## Section 3: Helper Functions

In [ ]:
def find_metrics_file(base_dir, seed):
    base = Path(base_dir)
    patterns = [
        base / f'seed_{seed}' / 'metrics.csv',
        base / f'seed{seed}' / 'metrics.csv',
        base / f'metrics_seed_{seed}.csv',
    ]
    for pattern in patterns:
        if pattern.exists():
            return pattern
    return None

def load_metrics_robust(metrics_file):
    if not metrics_file or not metrics_file.exists():
        return None
    df = pd.read_csv(metrics_file)
    if 'epoch' not in df.columns or 'val_acc' not in df.columns:
        return None
    if 'val_kl' not in df.columns:
        df['val_kl'] = np.nan
    return df

def collect_results(method_name, base_dir, seeds=[0, 1, 2]):
    results = []
    dataframes = []
    print(f"\nCollecting {method_name} results...")
    for seed in seeds:
        metrics_file = find_metrics_file(base_dir, seed)
        df = load_metrics_robust(metrics_file)
        if df is not None:
            final_acc = df['val_acc'].iloc[-1]
            results.append(final_acc)
            dataframes.append(df)
            print(f"  Seed {seed}: {final_acc:.2f}%")
        else:
            print(f"  Seed {seed}: ❌ Missing")
    return results, dataframes

print("✓ Helper functions defined")

## Section 4: Dense Students (3 Seeds)

In [ ]:
print("\n" + "="*70)
print("DENSE STUDENTS - 3 Seeds (~2-3 hours)")
print("="*70)

for seed in [0, 1, 2]:
    print(f"\n--- Seed {seed} ---")
    !python -m src.main dense --config configs/rq1/dense_multiclass.yaml --seed {seed} --out-dir runs/dense_multiclass/seed_{seed}

dense_results, dense_dfs = collect_results('Dense', 'runs/dense_multiclass')
if dense_results:
    print(f"\n✓ Dense: {np.mean(dense_results):.2f}% ± {np.std(dense_results):.2f}%")
else:
    print("\n❌ Dense training failed - check logs")

## Section 5: Progressive Students (3 Seeds)

In [ ]:
print("\n" + "="*70)
print("PROGRESSIVE STUDENTS - 3 Seeds (~2-3 hours)")
print("="*70)

for seed in [0, 1, 2]:
    print(f"\n--- Seed {seed} ---")
    !python -m src.main progressive --config configs/rq1/progressive_90pct_multiclass.yaml --seed {seed} --out-dir runs/progressive_90pct_multiclass/seed_{seed}

progressive_results, progressive_dfs = collect_results('Progressive', 'runs/progressive_90pct_multiclass')
if progressive_results:
    print(f"\n✓ Progressive: {np.mean(progressive_results):.2f}% ± {np.std(progressive_results):.2f}%")
else:
    print("\n❌ Progressive training failed - check logs")

## Section 6: One-Shot Students (3 Seeds)

In [ ]:
print("\n" + "="*70)
print("ONE-SHOT STUDENTS - 3 Seeds (~2-3 hours)")
print("="*70)

for seed in [0, 1, 2]:
    print(f"\n--- Seed {seed} ---")
    !python -m src.main oneshot_posthoc --config configs/rq1/oneshot_90pct_multiclass.yaml --seed {seed} --out-dir runs/oneshot_90pct_multiclass/seed_{seed}

oneshot_results, oneshot_dfs = collect_results('One-Shot', 'runs/oneshot_90pct_multiclass')
if oneshot_results:
    print(f"\n✓ One-Shot: {np.mean(oneshot_results):.2f}% ± {np.std(oneshot_results):.2f}%")
else:
    print("\n❌ One-Shot training failed - check logs")

## Section 7: Results Analysis

In [ ]:
print("\n" + "="*70)
print("FINAL RESULTS SUMMARY")
print("="*70)

all_complete = all([dense_results, progressive_results, oneshot_results])

results_data = {
    'Method': ['Teacher', 'Dense', 'Progressive (90%)', 'One-shot (90%)'],
    'Accuracy': [
        f"{teacher_acc:.2f}%",
        f"{np.mean(dense_results):.2f} ± {np.std(dense_results):.2f}%" if dense_results else "N/A",
        f"{np.mean(progressive_results):.2f} ± {np.std(progressive_results):.2f}%" if progressive_results else "N/A",
        f"{np.mean(oneshot_results):.2f} ± {np.std(oneshot_results):.2f}%" if oneshot_results else "N/A"
    ]
}

df_results = pd.DataFrame(results_data)
print("\n" + df_results.to_string(index=False))

if oneshot_results:
    gap = teacher_acc - np.mean(oneshot_results)
    print(f"\nGap (One-Shot to Teacher): {gap:.2f}%")

## Section 8: Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = {'Dense': '#2E86AB', 'Progressive': '#A23B72', 'One-shot': '#F18F01'}
methods = {'Dense': dense_dfs, 'Progressive': progressive_dfs, 'One-shot': oneshot_dfs}

for name, dfs in methods.items():
    if dfs:
        for i, df in enumerate(dfs):
            label = name if i == 0 else None
            ax1.plot(df['epoch'], df['val_acc'], color=colors[name], alpha=0.4, linewidth=1.5, label=label)

ax1.axhline(y=teacher_acc, color='black', linestyle='--', linewidth=2, label='Teacher')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Validation Accuracy (%)')
ax1.set_title('Convergence: Validation Accuracy (90% Sparsity)')
ax1.legend()
ax1.grid(True, alpha=0.3)

for name, dfs in methods.items():
    if dfs and 'val_kl' in dfs[0].columns and not dfs[0]['val_kl'].isna().all():
        for i, df in enumerate(dfs):
            label = name if i == 0 else None
            ax2.plot(df['epoch'], df['val_kl'], color=colors[name], alpha=0.4, linewidth=1.5, label=label)

ax2.set_xlabel('Epoch')
ax2.set_ylabel('KL Divergence')
ax2.set_title('Teacher Fidelity: KL Divergence')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('convergence_10class.png', dpi=300, bbox_inches='tight')
print("✓ Saved: convergence_10class.png")
plt.show()

## Section 9: Export and Archive

In [ ]:
with open('results_summary.txt', 'w') as f:
    f.write(df_results.to_string(index=False))

!cp convergence_10class.png /content/drive/MyDrive/ 2>/dev/null || true
!zip -qr results_10class.zip runs/ *.png *.txt *.csv 2>/dev/null || true
!cp results_10class.zip /content/drive/MyDrive/ 2>/dev/null || true

print("✓ Results exported and archived")

## Section 10: Final Report

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT STATUS")
print("="*70)

status = "✅ COMPLETE" if all_complete else "⚠️ PARTIAL"
print(f"\n{status}")
print(f"✓ Teacher: {teacher_acc:.2f}%")
if dense_results:
    print(f"✓ Dense: {np.mean(dense_results):.2f}%")
if progressive_results:
    print(f"✓ Progressive: {np.mean(progressive_results):.2f}%")
if oneshot_results:
    print(f"✓ One-Shot: {np.mean(oneshot_results):.2f}%")

if all_complete and oneshot_results:
    print(f"\n🎉 SUCCESS! Results ready for publication!")
    print(f"📁 All files saved to Google Drive")

print("\n" + "="*70)